# 0825_lsw_010_unsupervised_anomaly_detection

로드맵 Phase 4(비지도 이상탐지, "단순 이진분류기"라는 인상에서 벗어나는 확장 축). dongjin의
`0824_dongjin_015_isolation_forest`가 이미 단독 사용은 약하다는 걸 보였다(Test PR-AUC 0.036).
그래서 이번 노트북은 두 단계로 간다:

1. **단독 평가**: False call(정상, class=0) Train 데이터만으로 검사유형별 Isolation Forest를
   학습하고, 우리 표준 파이프라인(Slip Rate ≤1% 제약 하 threshold, Volume Reduction, 총비용)으로
   평가한다 — dongjin과 같은 결론이 우리 지표로도 재현되는지 확인.
2. **결합**: 이상탐지 점수를 XGBoost의 **추가 입력 피처**로 넣어서, 009까지 확정된 "현재 최고
   조합"(기법+튜닝된 하이퍼파라미터) 위에 이상 스코어 하나를 더했을 때 개선되는지 확인한다
   (로드맵 4-3, "현재 최고 유지 + 개선분 탐색" 방식 그대로).

Isolation Forest는 각 유형의 Train 중 **class=0(정상) 행만으로** 학습한다(비지도 이상탐지의
정의상 정상 분포만 학습). Validation/Test에는 그대로 점수만 매긴다(누수 없음).


## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_lsw_010_unsupervised_anomaly_detection"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)


experiment: 0825_lsw_010_unsupervised_anomaly_detection


## 2. 데이터 로딩·전처리 (003~009와 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]

timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
train_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))]
valid_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time


## 3. 평가 함수 (003~009와 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, score_val, max_slip_rate=0.01):
    """score가 높을수록 '불량 의심'이라고 가정 (predict_proba의 양성 확률과 같은 방향)."""
    candidates = np.sort(np.unique(score_val))[::-1]
    for t in candidates:
        y_pred = (score_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return float(np.min(score_val)) - 1.0  # 전부 1로 예측(안전 fallback)


def evaluate_at_threshold(y_true, score, threshold):
    y_pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, score),
        "roc_auc": roc_auc_score(y_true, score) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def resample_train(technique, X_train, y_train, n_pos):
    k_neighbors = max(1, min(5, n_pos - 1))
    if technique == "smote":
        return SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors).fit_resample(X_train, y_train)
    if technique == "undersample":
        return RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(X_train, y_train)
    return X_train, y_train


def build_xgb(**overrides):
    params = dict(
        random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
        objective="binary:logistic", eval_metric="logloss",
    )
    params.update(overrides)
    return XGBClassifier(**params)


## 4. 검사유형별 subset 및 baseline (5분리, 기본 하이퍼파라미터)

In [4]:
type_splits = {}
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]
    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)
    type_splits[inspection_type] = {
        "train": type_train_df, "valid": type_valid_df, "test": type_test_df,
        "feature_columns": type_feature_columns,
    }

baseline_results = {}
for inspection_type, split in type_splits.items():
    feature_columns = split["feature_columns"]
    model = build_xgb()
    model.fit(split["train"][feature_columns], split["train"][TARGET])
    valid_proba = model.predict_proba(split["valid"][feature_columns])[:, 1]
    threshold = select_threshold(split["valid"][TARGET], valid_proba)
    test_proba = model.predict_proba(split["test"][feature_columns])[:, 1]
    baseline_results[inspection_type] = evaluate_at_threshold(split["test"][TARGET], test_proba, threshold)

baseline_df = pd.DataFrame(baseline_results).T
baseline_df.index.name = "inspection_type"
baseline_df[["threshold", "tn", "fp", "fn", "tp", "total_cost_1:10", "total_cost_1:100"]]


,threshold,tn,fp,fn,tp,total_cost_1:10,total_cost_1:100
inspection_type,,,,,,,
0,0.000006,8857.0,7794.0,22.0,111.0,8014.0,9994.0
1,0.000025,1869.0,8459.0,8.0,776.0,8539.0,9259.0
2,0.000006,3492.0,14958.0,12.0,691.0,15078.0,16158.0
3,0.000007,9700.0,20288.0,43.0,560.0,20718.0,24588.0
4,0.000001,0.0,730.0,0.0,26.0,730.0,730.0


## 5. Isolation Forest 학습 (검사유형별, class=0만) — 단독 평가

Train 중 정상(class=0) 행만으로 학습한다. `score_samples`는 높을수록 "정상"이므로, 부호를
뒤집어 높을수록 "이상(불량 의심)"이 되도록 만든다.

In [5]:
iso_models = {}
iso_scores = {}  # (train_all, valid, test) 전체 행에 대한 이상 점수

for inspection_type, split in type_splits.items():
    feature_columns = split["feature_columns"]
    train_df = split["train"]
    normal_train_df = train_df.loc[train_df[TARGET] == 0]

    iso = IsolationForest(random_state=RANDOM_STATE, n_estimators=200, n_jobs=-1)
    iso.fit(normal_train_df[feature_columns])
    iso_models[inspection_type] = iso

    iso_scores[inspection_type] = {
        "train": -iso.score_samples(train_df[feature_columns]),
        "valid": -iso.score_samples(split["valid"][feature_columns]),
        "test": -iso.score_samples(split["test"][feature_columns]),
    }

iso_alone_results = {}
for inspection_type, split in type_splits.items():
    scores = iso_scores[inspection_type]
    threshold = select_threshold(split["valid"][TARGET], scores["valid"])
    iso_alone_results[inspection_type] = evaluate_at_threshold(split["test"][TARGET], scores["test"], threshold)

iso_alone_df = pd.DataFrame(iso_alone_results).T
iso_alone_df.index.name = "inspection_type"
iso_alone_df[["threshold", "tn", "fp", "fn", "tp", "pr_auc", "slip_rate", "volume_reduction", "total_cost_1:10", "total_cost_1:100"]]


,threshold,tn,fp,fn,tp,pr_auc,slip_rate,volume_reduction,total_cost_1:10,total_cost_1:100
inspection_type,,,,,,,,,,
0,0.388432,8715.0,7936.0,21.0,112.0,0.013745,0.157895,0.523392,8146.0,10036.0
1,0.391629,2864.0,7464.0,41.0,743.0,0.111821,0.052296,0.277304,7874.0,11564.0
2,0.392937,339.0,18111.0,0.0,703.0,0.052309,0.000000,0.018374,18111.0,18111.0
3,0.452275,9592.0,20396.0,53.0,550.0,0.020466,0.087894,0.319861,20926.0,25696.0
4,0.357373,6.0,724.0,0.0,26.0,0.098379,0.000000,0.008219,724.0,724.0


## 6. Isolation Forest 단독 vs XGBoost baseline 비교

In [6]:
compare_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    base = baseline_results[inspection_type]
    iso = iso_alone_results[inspection_type]
    compare_rows.append(
        {
            "inspection_type": inspection_type,
            "XGB_TN": base["tn"], "XGB_FN": base["fn"], "XGB_총비용(1:10)": base["total_cost_1:10"],
            "IsoForest_TN": iso["tn"], "IsoForest_FN": iso["fn"], "IsoForest_총비용(1:10)": iso["total_cost_1:10"],
            "IsoForest_PR-AUC": iso["pr_auc"], "XGB_PR-AUC": base["pr_auc"],
        }
    )
pd.DataFrame(compare_rows).set_index("inspection_type")


,XGB_TN,XGB_FN,XGB_총비용(1:10),IsoForest_TN,IsoForest_FN,IsoForest_총비용(1:10),IsoForest_PR-AUC,XGB_PR-AUC
inspection_type,,,,,,,,
0,8857,22,8014,8715,21,8146,0.013745,0.067265
1,1869,8,8539,2864,41,7874,0.111821,0.403144
2,3492,12,15078,339,0,18111,0.052309,0.356827
3,9700,43,20718,9592,53,20926,0.020466,0.269568
4,0,0,730,6,0,724,0.098379,0.023284


## 7. 결합 — 이상 점수를 XGBoost 추가 피처로 (009까지의 "현재 최고" 위에)

각 유형의 현재 최고 데이터 기법(004/007)과 009에서 찾은 튜닝 하이퍼파라미터를 그대로 쓰고,
Isolation Forest 이상 점수를 피처 하나 추가한 버전과 비교한다.

In [7]:
current_best_technique = {0: "label_cleansing", 1: "undersample", 2: "undersample", 3: "smote", 4: "undersample"}

# 009에서 확정된 튜닝 하이퍼파라미터 (비용비율별로 다를 수 있음, {} = 기본값)
tuned_hyperparams = {
    "1:10": {
        0: {},
        1: {"max_depth": 4, "min_child_weight": 20, "gamma": 0.0, "subsample": 1.0, "colsample_bytree": 0.8, "learning_rate": 0.03, "n_estimators": 300},
        2: {"max_depth": 4, "min_child_weight": 10, "gamma": 5.0, "subsample": 1.0, "colsample_bytree": 1.0, "learning_rate": 0.03, "n_estimators": 100},
        3: {"max_depth": 6, "min_child_weight": 10, "gamma": 3.0, "subsample": 1.0, "colsample_bytree": 0.6, "learning_rate": 0.3, "n_estimators": 100},
        4: {},
    },
    "1:100": {
        0: {},
        1: {},  # 009에서 1:100은 기본값이 더 나았음
        2: {"max_depth": 4, "min_child_weight": 10, "gamma": 5.0, "subsample": 1.0, "colsample_bytree": 1.0, "learning_rate": 0.03, "n_estimators": 100},
        3: {"max_depth": 6, "min_child_weight": 10, "gamma": 3.0, "subsample": 1.0, "colsample_bytree": 0.6, "learning_rate": 0.3, "n_estimators": 100},
        4: {},
    },
}


def apply_technique(train_df, feature_columns, technique):
    if technique == "label_cleansing":
        nunique_classes = train_df.groupby(feature_columns)[TARGET].transform("nunique")
        train_df = train_df.loc[~(nunique_classes > 1)]
        return train_df, train_df[feature_columns], train_df[TARGET]
    X_train, y_train = train_df[feature_columns], train_df[TARGET]
    n_pos = int((y_train == 1).sum())
    if technique in ("undersample", "smote"):
        X_res, y_res = resample_train(technique, X_train, y_train, n_pos)
        return None, X_res, y_res
    return train_df, X_train, y_train


champion_results = {}
champion_plus_anomaly_results = {}

for scenario in ["1:10", "1:100"]:
    for inspection_type, split in type_splits.items():
        technique = current_best_technique[inspection_type]
        feature_columns = split["feature_columns"]
        train_df, valid_df, test_df = split["train"], split["valid"], split["test"]

        # label_cleansing은 행이 줄어드는 것이라 이상점수 컬럼도 같이 잘라내야 함
        kept_train_df, X_res, y_res = apply_technique(train_df, feature_columns, technique)
        model_params = tuned_hyperparams[scenario][inspection_type]

        # --- (a) 현재 챔피언: 기법 + 튜닝 하이퍼파라미터, 이상점수 없음 ---
        model = build_xgb(**model_params)
        model.fit(X_res, y_res)
        valid_proba = model.predict_proba(valid_df[feature_columns])[:, 1]
        threshold = select_threshold(valid_df[TARGET], valid_proba)
        test_proba = model.predict_proba(test_df[feature_columns])[:, 1]
        champion_results[(inspection_type, scenario)] = evaluate_at_threshold(test_df[TARGET], test_proba, threshold)

        # --- (b) 챔피언 + 이상점수를 피처로 추가 ---
        anomaly_feature_columns = feature_columns + ["anomaly_score"]
        if technique == "label_cleansing":
            train_anomaly = -iso_models[inspection_type].score_samples(kept_train_df[feature_columns])
            X_res_anom = kept_train_df[feature_columns].copy()
            X_res_anom["anomaly_score"] = train_anomaly
            y_res_anom = kept_train_df[TARGET]
        else:
            # 리샘플링된 X_res는 원본 인덱스가 없을 수 있으므로 이상점수를 직접 계산
            train_anomaly = -iso_models[inspection_type].score_samples(X_res[feature_columns])
            X_res_anom = X_res.copy()
            X_res_anom["anomaly_score"] = train_anomaly
            y_res_anom = y_res

        valid_df_anom = valid_df[feature_columns].copy()
        valid_df_anom["anomaly_score"] = iso_scores[inspection_type]["valid"]
        test_df_anom = test_df[feature_columns].copy()
        test_df_anom["anomaly_score"] = iso_scores[inspection_type]["test"]

        model_anom = build_xgb(**model_params)
        model_anom.fit(X_res_anom[anomaly_feature_columns], y_res_anom)
        valid_proba_anom = model_anom.predict_proba(valid_df_anom[anomaly_feature_columns])[:, 1]
        threshold_anom = select_threshold(valid_df[TARGET], valid_proba_anom)
        test_proba_anom = model_anom.predict_proba(test_df_anom[anomaly_feature_columns])[:, 1]
        champion_plus_anomaly_results[(inspection_type, scenario)] = evaluate_at_threshold(
            test_df[TARGET], test_proba_anom, threshold_anom
        )

print("완료")


완료


## 8. 챔피언 vs 챔피언+이상점수 비교 (ΔTN/ΔFN, 총비용)

In [8]:
combo_rows = []
for scenario in ["1:10", "1:100"]:
    for inspection_type in sorted(clean_df["inspection_type"].unique()):
        champ = champion_results[(inspection_type, scenario)]
        combo = champion_plus_anomaly_results[(inspection_type, scenario)]
        combo_rows.append(
            {
                "inspection_type": inspection_type, "비용비율": scenario,
                "챔피언_TN": champ["tn"], "챔피언_FN": champ["fn"],
                "챔피언+이상점수_TN": combo["tn"], "챔피언+이상점수_FN": combo["fn"],
                "ΔTN": combo["tn"] - champ["tn"], "ΔFN": combo["fn"] - champ["fn"],
                "챔피언_총비용": champ[f"total_cost_{scenario}"],
                "챔피언+이상점수_총비용": combo[f"total_cost_{scenario}"],
                "개선": combo[f"total_cost_{scenario}"] - champ[f"total_cost_{scenario}"],
            }
        )
combo_df = pd.DataFrame(combo_rows).set_index(["inspection_type", "비용비율"])
combo_df


,,챔피언_TN,챔피언_FN,챔피언+이상점수_TN,챔피언+이상점수_FN,ΔTN,ΔFN,챔피언_총비용,챔피언+이상점수_총비용,개선
inspection_type,비용비율,,,,,,,,,
0,1:10,11433,46,9805,34,-1628,-12,5678,7186,1508
1,1:10,4815,36,4612,27,-203,-9,5873,5986,113
2,1:10,12934,80,12348,83,-586,3,6316,6932,616
3,1:10,20575,69,10081,35,-10494,-34,10103,20257,10154
4,1:10,15,0,3,0,-12,0,715,727,12
0,1:100,11433,46,9805,34,-1628,-12,9818,10246,428
1,1:100,4239,12,4231,18,-8,6,7289,7897,608
2,1:100,12934,80,12348,83,-586,3,13516,14402,886
3,1:100,20575,69,10081,35,-10494,-34,16313,23407,7094


## 9. 결론 및 다음 단계

### 방법론 노트 — 실수 하나 (기록)

처음 실행했을 때 type2만 이상점수 결합으로 개선되는 것처럼 보였다. 원인을 찾아보니 type2의
튜닝 하이퍼파라미터를 옮겨적으면서 **009 Phase B(리샘플링된 데이터에서 재탐색한 진짜 승자,
candidate 1)가 아니라 Phase A(baseline 데이터 기준 승자, candidate 15)의 설정을 잘못 복사**한
버그였다. 009에서 "baseline용 설정을 리샘플링 데이터에 재사용하면 안 된다"고 정리해놓고
바로 다음 노트북에서 비슷한 실수를 한 셈이다. `champion_results`가 009의 정답과 정확히
일치하는지 직접 재현해서 잡았고, 아래는 수정된 정확한 결과다.

### 단독 평가 (5절, dongjin의 `015_isolation_forest`와 같은 결론 재확인)

Isolation Forest 단독은 PR-AUC가 0.01~0.11 수준으로 XGBoost(0.02~0.40)보다 훨씬 약하다 —
dongjin의 "비지도 학습의 한계" 결론이 우리 지표로도 재현된다. 다만 흥미롭게도 **총비용
기준으로는 생각보다 덜 나쁘다**: type1(7,874 vs XGB 8,539)은 오히려 더 싸고, type0(8,146 vs
8,014)·type4(724 vs 730)는 거의 동률이다. type2(18,111 vs 15,078)만 뚜렷하게 나쁘다. PR-AUC가
훨씬 낮아도 Slip-Rate 제약 하 임계값 탐색이 그 차이를 어느 정도 흡수한다는, 이번에도 반복된
"PR-AUC ≠ 운영지표" 패턴의 또 다른 사례다.

### 결합 — 이상점수를 XGBoost 피처로 추가 (7~8절, 버그 수정 후 정답)

| type | 챔피언 총비용(1:10→1:100) | +이상점수 | 개선 |
|---|---|---|---:|
| 0 | 5,678 → 9,818 | 7,186 → 10,246 | +1,508 / +428 (악화) |
| 1 | 5,873 → 7,289 | 5,986 → 7,897 | +113 / +608 (악화) |
| 2 | 6,316 → 13,516 | 6,932 → 14,402 | +616 / +886 (악화) |
| 3 | 10,103 → 16,313 | 20,257 → 23,407 | +10,154 / +7,094 (크게 악화) |
| 4 | 715 → 715 | 727 → 727 | +12 / +12 (표본 부족) |

**전 유형·전 시나리오에서 예외 없이 악화됐다.** 이번 방식(Isolation Forest 원점수를 XGBoost
입력 피처 하나로 그냥 추가)은 이 데이터에서 통하지 않는다. 특히 type3(smote로 리샘플링된 데이터)
에서 가장 크게 나빠졌는데, SMOTE가 만든 합성 행들에 대해 원본 정상 분포로 학습한 Isolation
Forest 점수를 매기는 게 의미가 약했을 가능성이 크다 — 보간된 가짜 점에 이상 점수를 매기는 것
자체가 노이즈에 가깝다.

### 종합 결론

- **비지도 이상탐지 단독은 dongjin 결과대로 약하다** — 재확인.
- **이상 점수를 지도학습 피처로 얹는 이번 결합 방식도 통하지 않는다** — 009까지 쌓아온 "현재
  최고"를 어느 유형에서도 못 이겼다. Phase 4(이상탐지)는 "단순 분류기 이상의 시도를 했다"는
  스토리는 만들 수 있지만, **성능 개선으로는 이어지지 않았다**는 게 정직한 결론이다.
- 시도해볼 만했지만 안 한 결합 방식: (a) 이상 점수와 XGBoost 확률을 앙상블(가중 평균)하는
  방식, (b) 이상 점수가 매우 높은데 XGBoost는 정상이라 판단한 애매한 케이스만 추가로 수동
  검사에 보내는 규칙 기반 결합. 시간 관계상 이번엔 (a) 피처 추가 방식 하나만 검증했다.

### 다음 단계

1. 이 결과를 `docs/experiments/0825_lsw_010_unsupervised_anomaly_detection.md`로 옮기고
   `docs/experiments/index.md`, `handoff.md`에 반영한다.
2. 검사유형별 최종 조합표는 **004/007/009까지의 결과가 여전히 최종**이다(010에서 갱신되는 것
   없음).
3. 남은 옵션: 이상치 탐지 재구현(라벨 무관 극단값 기준, 여전히 미착수), 또는 여기서 실험을
   마무리하고 발표 자료로 결과를 정리하는 단계로 넘어가는 것.
